In [2]:
import sqlite3
import pandas as pd

# Kết nối đến cơ sở dữ liệu SQLite
conn = sqlite3.connect('db.sqlite3')
cursor = conn.cursor()
# danh sách các bảng trong sqlit3
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';") 
tables = cursor.fetchall()
# Truy vấn dữ liệu từ bảng
cursor.execute("SELECT * FROM CSDLHM_loaithucvat")
rows = cursor.fetchall()

# Lấy tên các cột
column_names = [description[0] for description in cursor.description]
# Chuyển đổi dữ liệu thành DataFrame
df_CDSL_Loai = pd.DataFrame(rows, columns=column_names).dropna(subset=['Tên_Loài_Tiếng_Việt'])
# Truy vấn dữ liệu từ bảng
cursor.execute("SELECT * FROM CSDLHM_herbalmedicine")
rows = cursor.fetchall()
# Lấy tên các cột
column_names = [description[0] for description in cursor.description]
# Chuyển đổi dữ liệu thành DataFrame
df_CSDLHM_herbalmedicine = pd.DataFrame(rows, columns=column_names).dropna(subset=['Dược_liệu_Tiếng_Việt'])
df_CSDLHM_herbalmedicine = df_CSDLHM_herbalmedicine.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df_CSDLHM_herbalmedicine['Dược_liệu_Tiếng_Latin_Kiểu_DĐVN'] = df_CSDLHM_herbalmedicine['Dược_liệu_Tiếng_Latin_Kiểu_DĐVN'].str.lower().str.replace(r'\s{2,}', ' ', regex=True).str.strip()
df_CSDL_SV = pd.read_excel('Duoc lieu.xlsx')
df_CSDL_SV = df_CSDL_SV.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df_CSDL_SV['Tên khoa học của Dược liệu'] = df_CSDL_SV['Tên khoa học của Dược liệu'].str.lower().str.replace(r'\s{2,}', ' ', regex=True).str.strip()
df_CSDL_SV['Tên tiếng việt dược liệu'] = df_CSDL_SV['Tên tiếng việt dược liệu'].str.lower().str.replace(r'\s{2,}', ' ', regex=True).str.strip().str.title()


In [3]:
name0=df_CSDLHM_herbalmedicine.columns[6]
name0
name1= df_CSDL_SV.columns[3]
df_CSDL_SV.rename(columns={name1: name0}, inplace=True)
# Sử dụng merge với how='outer'
df_CSDLHM = pd.merge(df_CSDLHM_herbalmedicine, df_CSDL_SV, on=[name0], how='right')

In [4]:
import os
import requests
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import DrawingOptions
import yaml
from pygbif import species
import requests
import json
from jsonpath_ng import jsonpath, parse
import io
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
import folium
from rdkit.Chem.Draw import rdMolDraw2D, rdDepictor
from rdkit.Chem.Draw import IPythonConsole
from IPython.display import SVG
from rdkit import rdBase
from rdkit.Chem.Draw import DrawingOptions
import base64
from io import BytesIO
from pygbif import species as species
from pygbif import occurrences as occ
from pygbif import maps
from mpl_toolkits.basemap import Basemap
from collections import defaultdict
import geopandas as gpd 
import folium
import matplotlib.pyplot as plt
import requests
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import DrawingOptions
import os
import requests
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import DrawingOptions
import os
import pandas as pd
from PIL import Image


# Hàm để chuyển đổi dòng DataFrame thành nội dung Markdown
def convert_row_to_markdown(row, columns):
    markdown_content = ""
    for col in columns:
        markdown_content += f"### {col}\n{row[col]}\n\n"
    return markdown_content

# Hàm để tạo cấu trúc hóa học và lưu hình ảnh
import os
import requests
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import DrawingOptions
from rdkit.Chem import AllChem
from rdkit.Chem import PandasTools
import os
import requests
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import DrawingOptions
from rdkit.Chem import PandasTools

def sanitize_filename(name):
    # Thay thế các ký tự không hợp lệ bằng dấu gạch dưới
    return "".join(c if c.isalnum() or c in " ._-" else "_" for c in name)

def get_chemical_structure_2(query, path):
    # Tạo đường dẫn tới API LOTUS
    url = f"https://lotus.naturalproducts.net/api/search/simple?query={query}"
    response = requests.get(url).json()
    # Chuyển đổi danh sách thành DataFrame
    compounds_df = pd.DataFrame(response['naturalProducts'])
    # Kiểm tra cột 'smiles' có tồn tại và không phải là None
    if 'smiles' not in compounds_df.columns or compounds_df['smiles'].isnull().all():
       raise KeyError("Cột 'smiles' không tồn tại hoặc không có dữ liệu hợp lệ trong DataFrame")
    compounds_df['smiles_count']=1
    compounds_df['name (code)'] = compounds_df.apply(lambda x: f"{x['traditional_name']} ({x['lotus_id']})", axis=1)

    grouped_compounds_df = compounds_df.groupby('chemicalTaxonomyClassyfireClass').agg({ 'smiles': lambda x: list(x), 'traditional_name': list, 'lotus_id': list, 'smiles_count': 'sum', 'name (code)': list }).reset_index()
    img_paths = []

    for _, row in grouped_compounds_df.iterrows(): 
        group = row['chemicalTaxonomyClassyfireClass'] 
        sanitized_group = sanitize_filename(group) 
        img_dir = os.path.join(path, query, "chemical structure") 
        if not os.path.exists(img_dir): 
            os.makedirs(img_dir)
        smiles_list =row['smiles'] 
        chembl_mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list if smiles]
        img_path = os.path.join(img_dir, f"{sanitized_group}.png")
        img = Draw.MolsToGridImage(chembl_mols, legends=row['lotus_id'], molsPerRow=4)
        with open(img_path, 'wb') as f: 
            f.write(img.data)
        img_paths.append(img_path)
    return compounds_df, grouped_compounds_df, img_paths
# Thông tin về phân bố trên thế giới cũng như phân bố tại việt nam 
from pygbif import species as species
from pygbif import occurrences as occ
from pygbif import maps
def get_list_images(query):
    try:
        spec = species.name_backbone(query)
        KeyGIBF = str(spec['usageKey'])
        path = f"https://api.gbif.org/v1/occurrence/search?taxonKey={KeyGIBF}&mediaType=StillImage&limit=5"
        response = requests.get(path).json()
        link_image = []
        for i in response['results']:
            for j in i['media']:
                if 'identifier' in j:
                    link_image.append(j['identifier'])
        return link_image
    except KeyError:
        print(f"Không tìm thấy 'usageKey' cho {query}")
        return []

def get_gibf(query):
    try:
        spec = species.name_backbone(query)
        return spec
    except KeyError:
        print("Không tìm thấy dữ liệu cho", query)
        return {}

def get_gibf_country(query):
        spec = species.name_backbone(query)
        if 'usageKey' in spec:
                occ_search = occ.search(taxonKey = spec['usageKey'])
                occ_df = pd.DataFrame(occ_search['results'])
                occ_global = ", ".join(str(country) for country in set(occ_df['country']) if country is not None) # phân bố trên thế giới
                filtered_df = occ_df[occ_df['country'] == 'Viet Nam'] # phân bố tại việt nam
                filtered_df['stateProvince'] = filtered_df['stateProvince'].fillna('')
                stateProvince_in_VN = ", ".join(str(state) for state in set(filtered_df['stateProvince']) if state)
                if not stateProvince_in_VN: 
                        stateProvince_in_VN = "Không có ghi nhận ở Việt Nam"
                return occ_global, stateProvince_in_VN
        else:
                return "Không có kết quả phù hợp", "Không có ghi nhận ở Việt Nam"

def get_first_words(text): 
    parts = text.split(' ') 
    if len(parts) >= 2: 
        return ' '.join(parts[:2]) 
    return

# Dược liệu

In [ ]:
#### folder dược liệu 
import os
from tabulate import tabulate
# Đảm bảo thư mục 'docs' tồn tại
if not os.path.exists('docs'):
    os.makedirs('docs')
# Đường dẫn tới thư mục lưu trữ các file markdown
path = "docs/Dược liệu/"
folder= "Dược liệu"
for idx, row in df_CSDLHM.iterrows():
    # Đảm bảo đường dẫn tới thư mục hình ảnh và file Markdown
    species_dir = os.path.join(path, row['Tên tiếng việt dược liệu'])
    if not os.path.exists(species_dir):
        os.makedirs(species_dir)
    DDHK_path = os.path.join(species_dir, "DDHK")
    SoiBot_path = os.path.join(species_dir, "Soi_Bot")
    ViPhau_path = os.path.join(species_dir, "Vi_Phau")
    os.makedirs(DDHK_path, exist_ok=True)
    os.makedirs(SoiBot_path, exist_ok=True)
    os.makedirs(ViPhau_path, exist_ok=True)
    Sci_name = get_first_words(row['Tên khoa học của Thực vật trong dược điển Việt Nam V'])
    Sci_name_gibf= get_gibf_country(Sci_name)
    try:
        #img_paths = get_chemical_structure(Sci_name, species_dir)
        compound_img= get_chemical_structure_2(Sci_name, species_dir)
    except KeyError as e:
        print(f"Không thể tạo cấu trúc hóa học cho {row['Tên tiếng việt dược liệu']}: {e}")
        #img_paths = ""  # Sử dụng một hình ảnh mặc định
        compound_img=""
    # Tạo nội dung Markdown cho các hình ảnh từ thư mục `images` 
    if compound_img:
        img_paths = [os.path.relpath(img, species_dir).replace("\\", "/") for img in compound_img[2]]
        list_compound = compound_img[1]
        markdown_table = tabulate(list_compound[['chemicalTaxonomyClassyfireClass','smiles_count']], headers='keys', tablefmt='pipe')
        def get_caption(img_name, df, col1, col2):
            match = df[df[col1] == img_name]
            if not match.empty:
                return match[col2].values[0]
            return "Không tìm thấy chú thích"

        chemical_images_2 = "\n".join([
            f'''
### Nhóm {os.path.basename(img).split('.')[0]}
<figure markdown="span">
    ![{os.path.basename(img)}]({img}){{ width=100% }}
    <figcaption>Hình ảnh cấu trúc hóa học của {get_caption(os.path.basename(img).split('.')[0], list_compound, 'chemicalTaxonomyClassyfireClass', 'smiles_count')} hoạt chất thuộc nhóm {os.path.basename(img).split('.')[0]} gồm {get_caption(os.path.basename(img).split('.')[0], list_compound, 'chemicalTaxonomyClassyfireClass', 'name (code)')}.</figcaption>
</figure>
        '''.strip() for img in img_paths
        ])

        count_comp = len(compound_img[0])
        list_comp = set(compound_img[0]['traditional_name'])
        # Lọc bỏ các giá trị None trước khi thực hiện join
        list_group_comp = ", ".join(filter(None, set(compound_img[0]['chemicalTaxonomyClassyfireClass'])))
    else:
        count_comp = "Chưa có hoạt chất nào được phân lập."
        chemical_images_2 = "Không có hình ảnh nào được tạo ra"
        list_comp = "Chưa có hoạt chất nào được phân lập"
        list_group_comp = ""
        markdown_table = ""

    # Lấy danh sách hình ảnh từ API GBIF
    image_links = get_list_images(Sci_name)
    image_markdown = "\n".join([f'<img src="{link}" alt="Mô tả hình ảnh" width="100" height="100">' for link in image_links[:7]])
    alternative_species_1 = "" 
    alternative_species_2 = "" 
    alternative_species_3 = "" 
    if not pd.isna(row['Loài thay thế 1']): 
        Sci_name_1 = get_first_words(row['Loài thay thế 1'])
        spec_l = get_gibf(Sci_name_1)
        image_links_1 = get_list_images(Sci_name_1)
        image_markdown_1 = "\n".join([f'<img src="{link}" alt="Mô tả hình ảnh" width="100" height="100">' for link in image_links_1[:7]])
        alternative_species_1 = f"""
Dược liệu này cũng có thể từ loài *{row['Loài thay thế 1']}*, thông tin về phân loại thực vật loài này như sau:
!!! info "Thông tin về phân loại thực vật của *{spec_l.get('species', 'N/A')}*"
    - **kingdom:** {spec_l.get('kingdom', 'N/A')}
    - **phylum:** {spec_l.get('phylum', 'N/A')}
    - **order:** {spec_l.get('order', 'N/A')}
    - **family:** {spec_l.get('family', 'N/A')}
    - **genus:** {spec_l.get('genus', 'N/A')}
    - **species:** *{spec_l.get('species', 'N/A')}*

Hình ảnh của loài *{row['Loài thay thế 1']}*:


{image_markdown_1}

"""

    if not pd.isna(row['Loài thay thế 2']): 
        Sci_name_2 = get_first_words(row['Loài thay thế 2'])
        spec_2 = get_gibf(Sci_name_2)
        image_links_2 = get_list_images(Sci_name_2)
        image_markdown_2 = "\n".join([f'<img src="{link}" alt="Mô tả hình ảnh" width="100" height="100">' for link in image_links_2[:7]])
        alternative_species_2 = f"""
Dược liệu này cũng có thể từ loài *{row['Loài thay thế 2']}*, thông tin về phân loại thực vật loài này như sau:
!!! info "Thông tin về phân loại thực vật của *{spec_2.get('species', 'N/A')}*"
    - **kingdom:** {spec_2.get('kingdom', 'N/A')}
    - **phylum:** {spec_2.get('phylum', 'N/A')}
    - **order:** {spec_2.get('order', 'N/A')}
    - **family:** {spec_2.get('family', 'N/A')}
    - **genus:** {spec_2.get('genus', 'N/A')}
    - **species:** *{spec_2.get('species', 'N/A')}*

Hình ảnh của loài *{row['Loài thay thế 2']}*:

{image_markdown_2}

"""


    if not pd.isna(row['Loài thay thế 3']): 
        Sci_name_3 = get_first_words(row['Loài thay thế 3'])
        spec_3 = get_gibf(Sci_name_3)
        image_links_3 = get_list_images(Sci_name_3)
        image_markdown_3 = "\n".join([f'<img src="{link}" alt="Mô tả hình ảnh" width="100" height="100">' for link in image_links_3[:7]])
        alternative_species_3 = f"""
Dược liệu này cũng có thể từ loài *{row['Loài thay thế 3']}*, thông tin về phân loại thực vật loài này như sau:
!!! info "Thông tin về phân loại thực vật của *{spec_3.get('species', 'N/A')}*"
    - **kingdom:** {spec_3.get('kingdom', 'N/A')}
    - **phylum:** {spec_3.get('phylum', 'N/A')}
    - **order:** {spec_3.get('order', 'N/A')}
    - **family:** {spec_3.get('family', 'N/A')}
    - **genus:** {spec_3.get('genus', 'N/A')}
    - **species:** *{spec_3.get('species', 'N/A')}*

Hình ảnh của loài *{row['Loài thay thế 3']}*:

{image_markdown_3}

"""
    # Lấy thông tin thực vật từ API GBIF
    spec = get_gibf(Sci_name)

    markdown_content = f"""---
title: {row['Dược_liệu_Tiếng_Việt']}
description: Dược liệu {row['Tên tiếng việt dược liệu']} từ bộ phận {row['Bộ_phận_dùng_Tiếng_Việt']} từ loài *{row['Tên khoa học của Thực vật trong dược điển Việt Nam V']}
date: 2024-12-01
categories:
  - {spec.get('family', 'N/A')}
tags:
  - {spec.get('family', 'N/A')}
  - {spec.get('genus', 'N/A')}
  - {spec.get('kingdom', 'N/A')}
---
!!! abstract "Tóm tắt"
    {row['Dựa tên thông tin thu thập được hãy viêt một đoạn tóm tắt ngắn về dược liệu']}

## Thông tin về thực vật

### Đặc điểm thực vật

Dược liệu **{row['Tên tiếng việt dược liệu']}** từ bộ phận **{row['Bộ_phận_dùng_Tiếng_Việt']}** từ loài *{row['Tên khoa học của Thực vật trong dược điển Việt Nam V']}* thuộc họ {spec.get('family', 'N/A')}. {row['Mô tả thực vật']} 

!!! info "Phân loại thực vật của *{spec.get('species', 'N/A')}*"
    - **Kingdom:** {spec.get('kingdom', 'N/A')}
    - **Phylum:** {spec.get('phylum', 'N/A')}
    - **Order:** {spec.get('order', 'N/A')}
    - **Family:** {spec.get('family', 'N/A')}
    - **Genus:** {spec.get('genus', 'N/A')}
    - **Species:** *{spec.get('species', 'N/A')}*

*Tài liệu tham khảo:* {row['Mô tả thực vật bạn sử dụng tài liệu nào']}


{image_markdown} 


### Loài thay thế (Nếu có)

{alternative_species_1}

{alternative_species_2}

{alternative_species_3}



### Phân bố trên thế giới
**Từ vườn thực vật KEW: **: {row['Phân bố của dược liệu trên thế giới ']}

**Từ CSDL GIBF** {Sci_name_gibf[0]}

### Phân bố tại Việt Nam
** {row['Mô tả thực vật bạn sử dụng tài liệu nào']}**: {row['Phân bố tại Việt Nam']}

**Từ CSDL GIBF**: {Sci_name_gibf[1]}

---

## Thông tin về dược liệu 

### Định danh

!!! info "Thông tin về tên gọi của {row['Dược_liệu_Tiếng_Việt']}"
    - Dược liệu tiếng Việt: {row['Dược_liệu_Tiếng_Việt']}
    - Dược liệu tiếng Trung: {row['Dược_liệu_Tiếng_Trung_Ký_tự']} ({row['Dược_liệu_Tiếng_Trung_Phiên_âm']})
    - Dược liệu tiếng Anh: {row['Dược_liệu_Tiếng_Anh']}
    - Dược liệu latin thông dụng: {row['Dược_liệu_Tiếng_Latin']}
    - Dược liệu latin kiểu DĐVN: {row['Dược_liệu_Tiếng_Latin_Kiểu_DĐVN']}
    - Dược liệu latin kiểu DĐVN: {row['Dược_liệu_Tiếng_Latin_Kiểu_DĐHK']}
    - Dược liệu latin kiểu thông tư: {row['Dược_liệu_Tiếng_Latin_Kiểu_Thông_Tư']}
    - Bộ phận dùng: {row['Bộ_phận_dùng_Tiếng_Việt']} ({row['Bộ_phận_dùng_Tiếng_Anh']})

### Mô tả dược liệu 
- **Theo dược điển Việt nam V:** {row['Dược_liệu_Mô_tả_DĐVN']}

- **Mô tả dược liệu theo thông tư chế biến dược liệu theo phương pháp cổ truyền:** {row['Dược_liệu_Mô_tả_Thông_Tư']}

### Chế biến 

- **Chế biến theo dược điển việt nam V**: {row['Bộ_phận_dùng_Chế_Biến_DĐVN']}

- **Chế biến theo thông tư:** {row['Bộ_phận_dùng_Chế_Biến_Thông_Tư']}

--- 

## Thành phần hóa học

- Theo tài liệu của GS. Đỗ Tất Lợi:  {row['Thành phần hóa học']}
    
- Theo cơ sở dữ liệu lotus: Từ loài *{spec.get('species', 'N/A')}* đã phân lập và xác định được {count_comp} hoạt chất thuộc về các nhóm {list_group_comp}. 

{markdown_table}

{chemical_images_2}

---

## Tác dụng dược lý

Theo tài liệu {row['Mô tả thực vật bạn sử dụng tài liệu nào']}:{row['Tác dụng dược lý của Dược liệu']}

Theo tài liệu quốc tế: {row['Dược_Liệu_Tác_Dụng_Quốc_tế_Usage']}

---

## Dược điển Việt Nam V

### Soi bột:
{row['Dược_liệu_Soi_Bột_DĐVN']}
<!-- Hình ảnh soi bột sẽ được tự động chèn vào đây sau -->
### Vi phẫu:
{row['Dược_liệu_Vi_Phẫu_DĐVN']}
<!-- Hình ảnh vi phẫu sẽ được tự động chèn vào đây sau -->
### Định tính

{row['Dược_liệu_Định_Tính_DĐVN']}

### Định lượng

{row['Dược_liệu_Định_Lượng_DĐVN']}

### Thông tin khác 
- ** Độ ẩm: ** {row['Dược_liệu_Độ_ẩm_DĐVN']}

- ** Bảo quản:** {row['Dược_liệu_Bảo_Quản_DĐVN']}
## Dược điển Hồng kong

<!-- PDF sẽ được tự động chèn vào đây sau -->


---

## Y dược học cổ truyền

- **Tên vị thuốc:** {row['Tên_Vị_Thuốc']}
- **Tính vị quy kinh:** {row['Tính vị quy kinh']}
- **Công năng chủ trị:** {row['Công năng chủ trị']}
- **Chú ý:** {row['Dược_liệu_Chú_Ý_DĐVN']}
- **Kiêng kỵ:** {row['Dược_liệu_Kiêng_Kỵ_DĐVN']}

"""
    file_name = f"{row['Tên tiếng việt dược liệu']}.md"
    file_path = os.path.join(species_dir, file_name)
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(markdown_content)

    # Cập nhật mkdocs.yml


print("Hoàn thành tạo các tệp Markdown và cập nhật mkdocs.yml.")


In [5]:
import os
import pandas as pd
from tabulate import tabulate
def generate_index_md(path, index_file='index.md'):
    # Lấy danh sách tất cả các tệp .md trong thư mục và các thư mục con
    md_files = []
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith('.md') and file != index_file:
                relative_path = os.path.relpath(os.path.join(root, file), path)
                md_files.append(relative_path)
                
    # Tạo DataFrame từ danh sách tệp Markdown
    data = {'title': [], 'md_file': []}
    for md_file in md_files:
        title = os.path.splitext(os.path.basename(md_file))[0]
        data['title'].append(title)
        data['md_file'].append(md_file)
    df = pd.DataFrame(data)
    return df




In [7]:

# Ví dụ sử dụng
path = '../docs/Dược liệu/'
# Tên tệp index.md

df_duoc_lieu_path = generate_index_md(path)
df_CSDLHM.rename(columns={'Tên tiếng việt dược liệu': df_duoc_lieu_path.columns[0]}, inplace=True)
df_CSDLHM.columns

Index(['page_ptr_id', 'Dược_liệu_Tiếng_Việt', 'Dược_liệu_Tiếng_Trung_Ký_tự',
       'Dược_liệu_Tiếng_Trung_Phiên_âm', 'Dược_liệu_Tiếng_Anh',
       'Dược_liệu_Tiếng_Latin', 'Dược_liệu_Tiếng_Latin_Kiểu_DĐVN',
       'Dược_liệu_Tiếng_Latin_Kiểu_DĐHK',
       'Dược_liệu_Tiếng_Latin_Kiểu_Thông_Tư', 'Bộ_phận_dùng_Tiếng_Việt',
       'Bộ_phận_dùng_Tiếng_Anh', 'Bộ_phận_dùng_Chế_Biến_DĐVN',
       'Bộ_phận_dùng_Chế_Biến_Thông_Tư', 'Dược_liệu_Mô_tả_DĐVN',
       'Dược_liệu_Mô_tả_Thông_Tư', 'Dược_liệu_Soi_Bột_DĐVN',
       'Dược_liệu_Vi_Phẫu_DĐVN', 'Dược_liệu_Định_Tính_DĐVN',
       'Dược_liệu_Định_Lượng_DĐVN', 'Dược_liệu_Liều_dùng_DĐVN',
       'Dược_liệu_Kiêng_Kỵ_DĐVN', 'Dược_liệu_Độ_ẩm_DĐVN',
       'Dược_liệu_Bảo_Quản_DĐVN', 'Dược_liệu_Chú_Ý_DĐVN',
       'Dược_Liệu_Tác_Dụng_Quốc_tế_Usage', 'Tên_Vị_Thuốc', 'Vị_Thuốc_Tính',
       'Vị_Thuốc_Vị', 'Vị_Thuốc_Quy_Kinh', 'Vị_Thuốc_Phân_Loại_Tác_dụng',
       'Dược_liệu_Công_Năng_Chủ_Trị_DĐVN', 'Tác_Dụng_Y_Dược_Cổ_Truyền',
       'Dược_Điển_Hồng_Kô

In [8]:

# Sử dụng merge với how='outer'
df_CSDLHM_with_path = pd.merge(df_CSDLHM, df_duoc_lieu_path, on=df_duoc_lieu_path.columns[0], how = "inner")
print(df_CSDLHM_with_path.columns)

Index(['page_ptr_id', 'Dược_liệu_Tiếng_Việt', 'Dược_liệu_Tiếng_Trung_Ký_tự',
       'Dược_liệu_Tiếng_Trung_Phiên_âm', 'Dược_liệu_Tiếng_Anh',
       'Dược_liệu_Tiếng_Latin', 'Dược_liệu_Tiếng_Latin_Kiểu_DĐVN',
       'Dược_liệu_Tiếng_Latin_Kiểu_DĐHK',
       'Dược_liệu_Tiếng_Latin_Kiểu_Thông_Tư', 'Bộ_phận_dùng_Tiếng_Việt',
       'Bộ_phận_dùng_Tiếng_Anh', 'Bộ_phận_dùng_Chế_Biến_DĐVN',
       'Bộ_phận_dùng_Chế_Biến_Thông_Tư', 'Dược_liệu_Mô_tả_DĐVN',
       'Dược_liệu_Mô_tả_Thông_Tư', 'Dược_liệu_Soi_Bột_DĐVN',
       'Dược_liệu_Vi_Phẫu_DĐVN', 'Dược_liệu_Định_Tính_DĐVN',
       'Dược_liệu_Định_Lượng_DĐVN', 'Dược_liệu_Liều_dùng_DĐVN',
       'Dược_liệu_Kiêng_Kỵ_DĐVN', 'Dược_liệu_Độ_ẩm_DĐVN',
       'Dược_liệu_Bảo_Quản_DĐVN', 'Dược_liệu_Chú_Ý_DĐVN',
       'Dược_Liệu_Tác_Dụng_Quốc_tế_Usage', 'Tên_Vị_Thuốc', 'Vị_Thuốc_Tính',
       'Vị_Thuốc_Vị', 'Vị_Thuốc_Quy_Kinh', 'Vị_Thuốc_Phân_Loại_Tác_dụng',
       'Dược_liệu_Công_Năng_Chủ_Trị_DĐVN', 'Tác_Dụng_Y_Dược_Cổ_Truyền',
       'Dược_Điển_Hồng_Kô

In [53]:

index_content = f""" 
!!! quote "Dược liệu là gì"
    Dược liệu là nguyên liệu làm thuốc có nguồn gốc tự nhiên từ thực vật, động vật, khoáng vật và đạt tiêu chuẩn làm thuốc.

    <div style="text-align: right;"> Mục 5, điều 2, Luật dược 2016 </div>
<div class="grid cards" markdown>
"""

for idx, row in df_CSDLHM_with_path.iterrows():
    summary = row['Dựa tên thông tin thu thập được hãy viêt một đoạn tóm tắt ngắn về dược liệu']
    name_files= row['title']
    md_file = row['md_file']
    # Tạo nội dung cho index.md
    index_content += f""" 

-   :material-leaf-circle-outline: __{name_files}__

    ---

    {summary}
    
    [{name_files}]({ md_file})



"""

index_content += f"""

</div>
"""

# Danh sách Dược liệu
path = '../docs/Dược liệu/'
index_file = "index.md"
 # Ghi nội dung vào index.md
with open(os.path.join(path, index_file), 'w', encoding='utf-8') as file:
    file.write(index_content)
    print(f"Hoàn thành tạo {index_file}")


# Đường dẫn tới thư mục Dược liệu


Hoàn thành tạo index.md


# Ethanopharmacology

In [9]:
import pandas as pd

# Đường dẫn tới tệp CSV
file_path_entha = "E:\\Database\\0-DataNaturalProduct\\dr-dukes-phytochemical-and-ethnobotanical-databases-All-2023-02-13_0033\\Duke-Source-CSV\\ETHNOBOT.csv"
file_path_activity = "E:\\Database\\0-DataNaturalProduct\\dr-dukes-phytochemical-and-ethnobotanical-databases-All-2023-02-13_0033\\Duke-Source-CSV\\ACTIVITIES.csv"

# Thử đọc tệp CSV với các mã hóa khác nhau
try:
    df_Entha = pd.read_csv(file_path_entha, encoding='utf-8')
except ValueError:
    df_Entha = pd.read_csv(file_path_entha, encoding='latin1')

try:
    df_Activity = pd.read_csv(file_path_activity, encoding='utf-8')
except ValueError:
    df_Activity = pd.read_csv(file_path_activity, encoding='latin1')

df_Entha = pd.merge(df_Entha, df_Activity, on="ACTIVITY", how = "inner", suffixes=('_df1', '_df2'))

In [10]:
# Nhóm dữ liệu theo các cột 'GENUS', 'SPECIES', 'FAMILY', 'COUNTRY' và tính gộp (aggregate) các cột khác
df_Entha_group = df_Entha.groupby(['GENUS', 'SPECIES', 'FAMILY', 'COUNTRY']).agg({
    'ACTIVITY': list,
    'DEFINITION': list
}).reset_index()
# Biến đổi các giá trị trong cột 'ACTIVITY' và 'DEFINITION_df1' từ dạng danh sách sang dạng chuỗi
df_Entha_group['ACTIVITY'] = df_Entha_group['ACTIVITY'].apply(lambda x: ', '.join(map(str, x)))
df_Entha_group['DEFINITION'] = df_Entha_group['DEFINITION'].apply(lambda x: ', '.join(map(str, x)))
df_Entha_group['Country_Enth']= df_Entha_group['COUNTRY']  + ": "+ df_Entha_group['DEFINITION']
df_Entha_group = df_Entha_group.groupby(['GENUS', 'SPECIES', 'FAMILY']).agg({
    'Country_Enth': list,
    'COUNTRY': set,
    'DEFINITION': set
}).reset_index()


In [11]:
import os
import pandas as pd
from translate import Translator
from tabulate import tabulate
def remove_invalid_chars(input_string):
    invalid_chars = "!@#$%^&*()_+-=[]{}|;:'\",.<>?/\\`~"
    translation_table = str.maketrans('', '', invalid_chars)
    return input_string.translate(translation_table)

In [12]:

# Đường dẫn tới thư mục gốc mới
root_directory_2 = r'\docs\Dược dân tộc'
# Tạo cột 'Đường dẫn mới' bằng cách kết hợp đường dẫn gốc mới với cột 'FAMILY'
df_Entha_group['Path_Family'] = df_Entha_group.apply(lambda row: os.path.join(root_directory_2, row['FAMILY']), axis=1)
df_Entha_group['Path_GENUS'] = df_Entha_group.apply(lambda row: os.path.join(root_directory_2, row['FAMILY'], row['GENUS']), axis=1)
df_Entha_group['Path_Species'] = df_Entha_group.apply(lambda row: os.path.join(root_directory_2, row['FAMILY'], row['GENUS'],remove_invalid_chars(str(row['GENUS'] + " " + row['SPECIES'])) ), axis=1)

In [20]:
df_Entha_group.head(3)

,GENUS,SPECIES,FAMILY,Country_Enth,COUNTRY,DEFINITION,Path_Family,Path_GENUS,Path_Species
0,Aaragus,adscendens,Liliaceae,"[Elsewhere: Refrigerant, Demulcent, India: Aph...","{India, Elsewhere, Iran}","{Refrigerant, Demulcent, Diaphoretic, Stimulan...",\docs\Dược dân tộc\Liliaceae,\docs\Dược dân tộc\Liliaceae\Aaragus,\docs\Dược dân tộc\Liliaceae\Aaragus\Aaragus a...
1,Aaragus,filicinus,Liliaceae,"[Elsewhere: Astringent, Tonic]",{Elsewhere},"{Astringent, Tonic}",\docs\Dược dân tộc\Liliaceae,\docs\Dược dân tộc\Liliaceae\Aaragus,\docs\Dược dân tộc\Liliaceae\Aaragus\Aaragus f...
2,Aaragus,lucidum,Liliaceae,"[China: Sedative, Tonic]",{China},"{Sedative, Tonic}",\docs\Dược dân tộc\Liliaceae,\docs\Dược dân tộc\Liliaceae\Aaragus,\docs\Dược dân tộc\Liliaceae\Aaragus\Aaragus l...


# Dược dân tộc học

In [13]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

def query_wikidata_for_species(species_name):
    """
    Truy vấn thông tin từ Wikidata về một loài thực vật dựa trên tên khoa học của loài.

    Args:
        species_name (str): Tên khoa học của loài thực vật.

    Returns:
        pd.DataFrame: DataFrame chứa các thông tin liên quan về loài từ Wikidata.
    """
    # Thiết lập endpoint SPARQL
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    # Truy vấn SPARQL để lấy thông tin về loài thực vật với nhãn tiếng Việt và tiếng Anh
    query = f"""
    SELECT ?item ?itemLabel ?itemLabel_vi ?itemLabel_en ?scientific_name 
           (GROUP_CONCAT(DISTINCT ?description_vi; separator=", ") AS ?descriptions_vi) 
           (GROUP_CONCAT(DISTINCT ?description_en; separator=", ") AS ?descriptions_en) 
           (GROUP_CONCAT(DISTINCT ?also_known_as_vi; separator=", ") AS ?also_known_as_vis) 
           (GROUP_CONCAT(DISTINCT ?also_known_as_en; separator=", ") AS ?also_known_as_ens) 
    WHERE {{
      ?item wdt:P31 wd:Q16521;  # Chỉ định loài thực vật
            wdt:P225 "{species_name}".  # Tên khoa học của loài thực vật
      OPTIONAL {{ ?item wdt:P225 ?scientific_name. }}
      OPTIONAL {{ ?item schema:description ?description_vi. FILTER(LANG(?description_vi) = "vi") }}  # Mô tả bằng tiếng Việt
      OPTIONAL {{ ?item schema:description ?description_en. FILTER(LANG(?description_en) = "en") }}  # Mô tả bằng tiếng Anh
      OPTIONAL {{ ?item skos:altLabel ?also_known_as_vi. FILTER(LANG(?also_known_as_vi) = "vi") }}  # Tên khác bằng tiếng Việt
      OPTIONAL {{ ?item skos:altLabel ?also_known_as_en. FILTER(LANG(?also_known_as_en) = "en") }}  # Tên khác bằng tiếng Anh
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],vi,en". }}
    }}
    GROUP BY ?item ?itemLabel ?itemLabel_vi ?itemLabel_en ?scientific_name
    """

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    # Chuyển kết quả truy vấn thành DataFrame
    data = []
    for result in results["results"]["bindings"]:
        item = result["item"]["value"]
        itemLabel_vi = result.get("itemLabel_vi", {}).get("value", "N/A")
        itemLabel_en = result.get("itemLabel_en", {}).get("value", "N/A")
        scientific_name = result.get("scientific_name", {}).get("value", "N/A")
        descriptions_vi = result.get("descriptions_vi", {}).get("value", "N/A")
        descriptions_en = result.get("descriptions_en", {}).get("value", "N/A")
        also_known_as_vis = result.get("also_known_as_vis", {}).get("value", "N/A").split(", ")
        also_known_as_ens = result.get("also_known_as_ens", {}).get("value", "N/A").split(", ")
        data.append({
            "Item": item,
            "Label (VI)": itemLabel_vi,
            "Label (EN)": itemLabel_en,
            "Scientific Name": scientific_name,
            "Descriptions (VI)": descriptions_vi,
            "Descriptions (EN)": descriptions_en,
            "Also Known As (VI)": also_known_as_vis,
            "Also Known As (EN)": also_known_as_ens
        })

    df_species = pd.DataFrame(data)

    return df_species




In [ ]:
import os
import pandas as pd
from translate import Translator
# Đảm bảo thư mục 'docs' tồn tại
path = "docs"
folder = "Dược dân tộc"
if not os.path.exists(path):
    os.makedirs(path)
folder_names_duoc_dan_toc = set([name for name in os.listdir(os.path.join(path, folder)) if os.path.isdir(os.path.join(path, folder, name))])
# Tạo DataFrame từ danh sách tên folder

FAMILY_SET = set(df_Entha_group["FAMILY"])
filtered_family_set = FAMILY_SET - folder_names_duoc_dan_toc
print('Số học còn lại:', len(filtered_family_set))
for Family in filtered_family_set:
    family_folder = os.path.join(path, folder, Family)
    if not os.path.exists(family_folder):
        os.makedirs(family_folder)
    filtered_df_Entha_group_family = df_Entha_group.loc[df_Entha_group['FAMILY'] == Family]
    tags_spec = "\n".join([f"  - {genus} {species}" for genus, species in zip(filtered_df_Entha_group_family["GENUS"], filtered_df_Entha_group_family["SPECIES"])])
    tags_genus = "\n".join([f"  - {genus}" for genus in filtered_df_Entha_group_family["GENUS"]])
    
    list_country_family = set()
    for countries in filtered_df_Entha_group_family["COUNTRY"]:
        if isinstance(countries, set):
            list_country_family.update(countries)
        else:
            list_country_family.add(countries)
    list_country_family = ', '.join(list_country_family)
    list_disease_family = set()
    for disease in filtered_df_Entha_group_family["DEFINITION"]:
        if isinstance(disease, set):
            list_disease_family.update(disease)
        else:
            list_disease_family.add(disease)
    list_disease_family = ', '.join(list_disease_family)
    Trans = Translator(to_lang='vi', from_lang='en')
    list_disease_family_vi = Trans.translate(list_disease_family) 
    markdown_content = f"""---
title: Các dược liệu thuộc họ {Family}
description: Họ {Family} gồm khoảng {filtered_df_Entha_group_family["GENUS"].nunique()} chi và {filtered_df_Entha_group_family["SPECIES"].nunique()} loài được một số cộng đồng tại các quốc gia như {list_country_family} sử dụng trong hỗ trợ và điều trị bệnh {list_disease_family_vi}.
date: 2024-12-01
categories:
  - Dược dân tộc
  - {Family}
tags:
  - Dược dân tộc
{tags_spec}
{tags_genus}
---
!!! abstract "Tóm tắt"

    Họ {Family} gồm khoảng {filtered_df_Entha_group_family["GENUS"].nunique()} chi và {filtered_df_Entha_group_family["SPECIES"].nunique()} loài được một số cộng đồng tại các quốc gia như {list_country_family} sử dụng trong một số trường hợp {list_disease_family_vi}.

!!! info "DrDuke"

    James A. Duke sinh năm 1929-2017 là một nhà thực vật học người Mỹ. Đây là một trong những tác giả hàng đầu trong lĩnh vực dược dân tộc học với cuốn *CRC Handbook of Medicinal Herbs* và chính là người xây dựng lên cơ sở dữ liệu về hợp chất tự nhiên và dược dân tộc học tại Bộ nông nghiệp Hoa Kỳ. Các thông tin được đăng tải tại website [Dr. Duke's Phytochemical and Ethnobotanical Databases](https://phytochem.nal.usda.gov/). 
    Trong suốt thập niên 1970, ông lãnh đạo the Plant Taxonomy Laboratory, Plant Genetics and Germplasm Institute of the Agricultural Research Service, U.S. Department of Agriculture.
    Trong tài liệu này, các thông tin về dược dân tộc của các dược liệu được trích dẫn từ tài liệu của James A. Ducke với sự trợ giúp của phần mềm dịch thuật từ tiếng Anh sang tiếng Việt.
   
"""

    GENUS_SET = set(filtered_df_Entha_group_family["GENUS"])
    for genus in GENUS_SET:
        genus_folder = os.path.join(path, folder, Family, genus)
        if not os.path.exists(genus_folder):
            os.makedirs(genus_folder)    
        filtered_df_Entha_group_genus = filtered_df_Entha_group_family.loc[filtered_df_Entha_group_family['GENUS'] == genus]
        set_spec_in_genus = "\n".join([f"\t - *{genus} {species}*" for genus, species in zip(filtered_df_Entha_group_genus["GENUS"], filtered_df_Entha_group_genus["SPECIES"])])

        markdown_content += f"""
# Chi {genus}

??? note "Danh sách các dược liệu thuộc chi"
    
{set_spec_in_genus}

"""
        for idx, row in filtered_df_Entha_group_genus.iterrows():
            try:
                Sci_name = str(row['GENUS'] + " " + row['SPECIES'])
                Sci_name = remove_invalid_chars(Sci_name)
                species_dir = os.path.join(path, folder, Family, genus, Sci_name)
                if not os.path.exists(species_dir):
                    os.makedirs(species_dir)
                # Chuẩn bị dữ liệu Dược dân tộc học
                Coutry_enth = ';'.join(row['Country_Enth'])
                pairs = [item.strip() for item in Coutry_enth.split(';')]
                data_Coutry_enth = {'Country': [], 'Disease': [], 'Bệnh': []}
                translator = Translator(to_lang='vi', from_lang='en')
                for pair in pairs: 
                    Country, Disease = pair.split(':') 
                    data_Coutry_enth['Country'].append(Country.strip()) 
                    data_Coutry_enth['Disease'].append(Disease.strip()) 
                    translated_disease = translator.translate(Disease.strip()) 
                    data_Coutry_enth['Bệnh'].append(translated_disease)
                data_Coutry_enth = pd.DataFrame(data_Coutry_enth)
                markdown_table_data_Coutry_enth = data_Coutry_enth.to_markdown(index=False)
                try: 
                    Sci_name_gibf = get_gibf_country(Sci_name) 
                except KeyError as e: 
                    print(f"Lỗi KeyError: {e}. Kiểm tra lại cấu trúc dữ liệu trả về từ hàm get_gibf_country.")
                try:
                    compound_img = get_chemical_structure_2(Sci_name, genus_folder)
                except KeyError as e:
                    print(f"Không thể tạo cấu trúc hóa học: {e}")
                    compound_img = ""
                if compound_img:
                    img_paths = [os.path.relpath(img, family_folder).replace("\\", "/") for img in compound_img[2]]               
                    list_compound = compound_img[1]
                    markdown_table = tabulate(list_compound[['chemicalTaxonomyClassyfireClass', 'smiles_count']], headers='keys', tablefmt='pipe')

                    def get_caption(img_name, df, col1, col2):
                        match = df[df[col1] == img_name]
                        if not match.empty:
                            return match[col2].values[0]
                        return "Không tìm thấy chú thích"
                    chemical_images_2 = "\n".join([f'''
#### Nhóm {os.path.basename(img).split('.')[0]}
<figure markdown="span">
    ![{os.path.basename(img)}]({img}){{ width=100% }}
    <figcaption>Hình ảnh cấu trúc hóa học của {get_caption(os.path.basename(img).split('.')[0], list_compound, 'chemicalTaxonomyClassyfireClass', 'smiles_count')} hoạt chất thuộc nhóm {os.path.basename(img).split('.')[0]} gồm {get_caption(os.path.basename(img).split('.')[0], list_compound, 'chemicalTaxonomyClassyfireClass', 'name (code)')}.</figcaption>
</figure>
'''.strip() for img in img_paths])

                    count_comp = len(compound_img[0])
                    list_comp = set(compound_img[0]['traditional_name'])
                    list_group_comp = ", ".join(filter(None, set(compound_img[0]['chemicalTaxonomyClassyfireClass'])))
                else:
                    count_comp = "Chưa có hoạt chất nào được phân lập."
                    chemical_images_2 = "Không có hình ảnh nào được tạo ra"
                    list_comp = "Chưa có hoạt chất nào được phân lập"
                    list_group_comp = "Không có hoạt chất nào được phân lập"
                    markdown_table = ""

                image_links = get_list_images(Sci_name)
                image_markdown = "\n".join([f'<img src="{link}" alt="Mô tả hình ảnh" width="100" height="100">' for link in image_links[:7]])
                spec = get_gibf(Sci_name)
                df_wikidata = query_wikidata_for_species(Sci_name)

                if 'Item' in df_wikidata.columns:
                    caption_table_df_wikidata = df_wikidata["Item"]
                    data_table_df_wikidata = df_wikidata.drop(columns=['Item'])
                    markdown_table_df_wikidata = data_table_df_wikidata.to_markdown(index=False)
                else:
                    print("Column 'Item' không tồn tại trong DataFrame.")
                markdown_content += f"""---
## {Sci_name}
### Thông tin về thực vật

!!! info "Phân loại thực vật của *{spec.get('species', 'N/A')}* từ GIBF:"
    - **Kingdom:** {spec.get('kingdom', 'N/A')}
    - **Phylum:** {spec.get('phylum', 'N/A')}
    - **Order:** {spec.get('order', 'N/A')}
    - **Family:** {spec.get('family', 'N/A')}
    - **Genus:** {spec.get('genus', 'N/A')}
    - **Species:** *{spec.get('species', 'N/A')}*

{image_markdown} 

![Nguồn thông tin từ wiki](caption_table_df_wikidata)

{markdown_table_df_wikidata}

#### Phân bố trên thế giới

**Từ CSDL GIBF** {Sci_name_gibf[0]}

#### Phân bố tại Việt Nam

**Từ CSDL GIBF**: {Sci_name_gibf[1]}

---
### Thành phần hóa học
        
- Theo cơ sở dữ liệu lotus: Từ loài *{spec.get('species', 'N/A')}* đã phân lập và xác định được {count_comp} hoạt chất thuộc về các nhóm {list_group_comp}. 

{markdown_table}

{chemical_images_2}

---

### Dược dân tộc học

Danh sách các quốc gia có sử dụng *{spec.get('species', 'N/A')}* trong điều trị các bệnh. 

{markdown_table_data_Coutry_enth}
\n
---

"""
            except Exception as e: 
                print(f"Lỗi tại dòng {idx}: {e}") # Tiếp tục vòng lặp với dòng tiếp theo continue    

    # Đảm bảo rằng các thư mục của FAMILY tồn tại trước khi tạo file

    
    file_name = f"{Family}.md"
    file_path = os.path.join(family_folder, file_name)
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(markdown_content)
    print(f"Hoàn thành tạo {file_path}")


In [43]:
df_Entha_group.head(3)

,GENUS,SPECIES,FAMILY,Country_Enth,COUNTRY,DEFINITION,Path_Family,Path_GENUS,Path_Species,file_name,file_paths
0,Aaragus,adscendens,Liliaceae,"[Elsewhere: Refrigerant, Demulcent, India: Aph...","{Elsewhere, India, Iran}","{Diaphoretic, Stimulant, Aphrodisiac, Tonic, R...",\docs\Dược dân tộc\Liliaceae,\docs\Dược dân tộc\Liliaceae\Aaragus,\docs\Dược dân tộc\Liliaceae\Aaragus\Aaragus a...,Liliaceae,Dược dân tộc/Liliaceae/Liliaceae.md
1,Aaragus,filicinus,Liliaceae,"[Elsewhere: Astringent, Tonic]",{Elsewhere},"{Astringent, Tonic}",\docs\Dược dân tộc\Liliaceae,\docs\Dược dân tộc\Liliaceae\Aaragus,\docs\Dược dân tộc\Liliaceae\Aaragus\Aaragus f...,Liliaceae,Dược dân tộc/Liliaceae/Liliaceae.md
2,Aaragus,lucidum,Liliaceae,"[China: Sedative, Tonic]",{China},"{Sedative, Tonic}",\docs\Dược dân tộc\Liliaceae,\docs\Dược dân tộc\Liliaceae\Aaragus,\docs\Dược dân tộc\Liliaceae\Aaragus\Aaragus l...,Liliaceae,Dược dân tộc/Liliaceae/Liliaceae.md


In [15]:
import os
import pandas as pd



# Nội dung Markdown cho index.md
index_content = """ 
!!! quote "Dược dân tộc học là gì"
    Dược dân tộc học (Ethnopharmacology) là thuật ngữ xuất hiện năm 1967 được hiểu là môn khoa học nghiên cứu về các nguyên liệu một nhóm dân tộc sử dụng như thuốc. Nghiên cứu thuốc cổ truyền, kinh nghiệm dân gian có thể hiểu là một phần dược dân tộc nhưng nếu đặt trong ví dụ về qua trình kinh nghiệm sử dụng một loài dược liệu xuất hiện và truyền miệng trong cộng đồng dù trước đó chưa được biết tới do giao thương văn hóa làm nổi bật sự khác biệt của dược dân tộc học. Dược dân tộc học thường đề cập tới thực vật bậc cao do đó đây là nhánh của thực vật dân tộc học - chuyên nghiên cứu về ứng dụng của một loài thực vật trong đời sống hàng ngày của cộng đồng đơn giản như cỏ tranh được sử dụng để lợp mái nhà, hoặc cây sơn để làm sơn mài. Một nghiên cứu dược dân tộc học yêu cầu tập hợp kiến thức các chuyên ngành khác nhau thường bắt đầu từ điều tra kinh nghiệm sử dụng nguyên liệu làm thuốc trong không gian cộng đồng và kết thúc là nghiên cứu cung cấp bằng chứng khoa học trong phòng thí nghiệm. Dược dân tộc học đóng vai trò quan trọng chỉ dẫn đối tượng nghiên cứu trong phát triển thuốc từ thảo mộc.    
<div class="grid cards" markdown>
"""

# Khởi tạo giá trị chương ban đầu
chapter_number = 1

# Tạo nội dung cho update_index và index.md
update_index = "- Dược dân tộc:\n"

df_Entha_group_index = df_Entha_group.groupby(['FAMILY']).agg({'GENUS': set}).reset_index()

path = "..\\docs"
folder = "Dược dân tộc"

df_Entha_group_index['file_name'] = df_Entha_group_index["FAMILY"]
df_Entha_group_index["file_paths"] = df_Entha_group_index.apply(lambda row: os.path.join(folder, row['FAMILY'], f"{row['FAMILY']}.md").replace("\\", "/"), axis=1)

# Lặp qua các hàng trong DataFrame và tạo nội dung cho các tệp
for _, row in df_Entha_group_index.iterrows():
    family = row['FAMILY']
    genus_str = ', '.join(row['GENUS'])
    file_name = f"{family}.md"
    file_path = os.path.join(folder, family, file_name).replace("\\", "/")
    
    update_index += f"  - {family}: {file_path}\n"
    
    index_content += f"""
-   :material-leaf-circle-outline: __{family}__

    ---

    Các chi thuộc họ {family} được sử dụng làm thuốc gồm {genus_str}.
    [Họ {family}]({file_path})

"""

index_content += """
</div>
"""

# Ghi nội dung vào tệp index.md
index_file_path = os.path.join(path, folder, "index.md")
with open(index_file_path, 'w', encoding='utf-8') as file:
    file.write(index_content)
print(f"Hoàn thành tạo {index_file_path}")

# Ghi nội dung vào tệp update_index.yml
update_index_file_path = os.path.join(path, folder, "update_index.yml")
with open(update_index_file_path, 'w', encoding='utf-8') as file:
    file.write(update_index)
print(f"Hoàn thành tạo {update_index_file_path}")


Hoàn thành tạo ..\docs\Dược dân tộc\index.md
Hoàn thành tạo ..\docs\Dược dân tộc\update_index.yml


In [64]:
import os
import re

def extract_species(content):
    # Tìm tên loài trong nội dung của file bằng regex
    match = re.search(r"'scientificName': '(.*?)'", content)
    if match:
        return match.group(1)
    return None

def update_md_files(directory):
    # Duyệt qua tất cả các thư mục và file trong thư mục gốc
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith('.md'):
                filepath = os.path.join(root, filename)
                with open(filepath, 'r', encoding='utf-8') as file:
                    content = file.read()

                # Trích xuất tên loài từ nội dung file
                species_name = extract_species(content)
                if species_name:
                    # Tạo tiêu đề mới với tên loài
                    new_title = f"*{species_name}*"
                    new_description=f"*{species_name}* được sử dụng trong nhiều nền văn hóa khác nhau."
                    
                    # Thay thế phần title và description hiện tại
                    new_content = re.sub(r"title: {.*?}", f"title: {new_title}", content, flags=re.DOTALL)
                    new_content = re.sub(r"description: {.*?}", f"description: {new_description}", new_content, flags=re.DOTALL)
                    new_content = re.sub(r" được sử dụng trong y học dân tộc nhiều quốc gia.", "", content)

                    # Thay thế đoạn !!! info "Phân loại thực vật của *Abrus precatorius* từ GIBF\n"
                    new_content = re.sub(r'từ GIBF', 'từ GIBF\n', new_content, flags=re.DOTALL)

                    # Thay thế đoạn !!! info "DrDuke"\n
                    new_content = re.sub(r'!!! info "DrDuke"', '!!! info "DrDuke"\n', new_content, flags=re.DOTALL)

                    with open(filepath, 'w', encoding='utf-8') as file:
                        file.write(new_content)



In [65]:
# Đường dẫn tới thư mục chứa các file md


update_md_files(directory)


In [10]:
import os
import pandas as pd

def list_first_level_folders(root_directory):
    folder_names = []
    folder_paths = []

    # Duyệt qua tất cả các thư mục và file trong thư mục gốc
    for root, dirs, files in os.walk(root_directory):
        for dir_name in dirs:
            # Thêm tên folder và đường dẫn đầy đủ của thư mục bậc 1
            folder_names.append(dir_name)
            folder_paths.append(os.path.join(root, dir_name))
        break  # Chỉ duyệt qua thư mục gốc, không đi sâu hơn

    # Tạo DataFrame từ danh sách tên folder và đường dẫn
    df = pd.DataFrame({
        'FAMILY': folder_names,
        'Đường dẫn cũ': folder_paths
    })

    return df

# Đường dẫn tới thư mục gốc
root_directory = r'..\docs\Dược dân tộc'

# Tạo DataFrame
df_path_enth = list_first_level_folders(root_directory)

# Hiển thị DataFrame
print(df_path_enth)



              FAMILY                          Đường dẫn cũ
0        Acanthaceae      ..\docs\Dược dân tộc\Acanthaceae
1          Aceraceae        ..\docs\Dược dân tộc\Aceraceae
2    Achatocarpaceae  ..\docs\Dược dân tộc\Achatocarpaceae
3          Acoraceae        ..\docs\Dược dân tộc\Acoraceae
4      Actinidiaceae    ..\docs\Dược dân tộc\Actinidiaceae
..               ...                                   ...
270      Winteraceae      ..\docs\Dược dân tộc\Winteraceae
271       Xyridaceae       ..\docs\Dược dân tộc\Xyridaceae
272        Zamiaceae        ..\docs\Dược dân tộc\Zamiaceae
273    Zingiberaceae    ..\docs\Dược dân tộc\Zingiberaceae
274   Zygophyllaceae   ..\docs\Dược dân tộc\Zygophyllaceae

[275 rows x 2 columns]


In [11]:
df_Entha_path_fix = pd.merge(df_Entha_group, df_path_enth, on="FAMILY", how = "inner", suffixes=('_df1', '_df2'))
df_Entha_path_fix


,GENUS,SPECIES,FAMILY,Country_Enth,Đường dẫn cũ
0,Aaragus,adscendens,Liliaceae,"[Elsewhere: Refrigerant, Demulcent, India: Aph...",..\docs\Dược dân tộc\Liliaceae
1,Aaragus,filicinus,Liliaceae,"[Elsewhere: Astringent, Tonic]",..\docs\Dược dân tộc\Liliaceae
2,Aaragus,lucidum,Liliaceae,"[China: Sedative, Tonic]",..\docs\Dược dân tộc\Liliaceae
3,Aaragus,lucidus,Liliaceae,"[China: Diuretic, Expectorant, Expectorant, Ne...",..\docs\Dược dân tộc\Liliaceae
4,Aaragus,officinalis,Liliaceae,"[China: Diuretic, Parasiticide, Dominican Repu...",..\docs\Dược dân tộc\Liliaceae
...,...,...,...,...,...
6436,ondias,lutea,Anacardiaceae,"[Mexico: Astringent, Venezuela: Astringent, Ci...",..\docs\Dược dân tộc\Anacardiaceae
6437,ondias,mombin,Anacardiaceae,"[Dominican Republic: Poison, Elsewhere: Antise...",..\docs\Dược dân tộc\Anacardiaceae
6438,ondias,pinnata,Anacardiaceae,"[Elsewhere: Antidote, Astringent, Demulcent]",..\docs\Dược dân tộc\Anacardiaceae
6439,ondias,purpurea,Anacardiaceae,"[Dominican Republic: Laxative, Vulnerary, Into...",..\docs\Dược dân tộc\Anacardiaceae


In [ ]:
Không chạy nữa 

In [138]:
# không chạy nữua
import os
import pandas as pd

# Đường dẫn tới thư mục gốc mới
root_directory_2 = r'..\docs\Dược dân tộc 2'

# Tạo cột 'Đường dẫn mới' bằng cách kết hợp đường dẫn gốc mới với cột 'FAMILY'
df_Entha_path_fix['Đường dẫn mới'] = df_Entha_path_fix.apply(lambda row: os.path.join(root_directory_2, row['FAMILY'], row['GENUS']), axis=1)

# In ra các cột 'Đường dẫn cũ' và 'Đường dẫn mới'
print(df_Entha_path_fix[['Đường dẫn cũ', 'Đường dẫn mới']])


                          Đường dẫn cũ  \
0     ..\docs\Dược dân tộc\Abelmoschus   
1     ..\docs\Dược dân tộc\Abelmoschus   
2     ..\docs\Dược dân tộc\Abelmoschus   
3           ..\docs\Dược dân tộc\Abies   
4           ..\docs\Dược dân tộc\Abies   
...                                ...   
6775       ..\docs\Dược dân tộc\Zornia   
6776     ..\docs\Dược dân tộc\Zuelania   
6777     ..\docs\Dược dân tộc\Zuelania   
6778  ..\docs\Dược dân tộc\Zygophyllum   
6779  ..\docs\Dược dân tộc\Zygophyllum   

                                          Đường dẫn mới  
0          ..\docs\Dược dân tộc 2\Malvaceae\Abelmoschus  
1          ..\docs\Dược dân tộc 2\Malvaceae\Abelmoschus  
2          ..\docs\Dược dân tộc 2\Malvaceae\Abelmoschus  
3                 ..\docs\Dược dân tộc 2\Pinaceae\Abies  
4                 ..\docs\Dược dân tộc 2\Pinaceae\Abies  
...                                                 ...  
6775             ..\docs\Dược dân tộc 2\Fabaceae\Zornia  
6776     ..\docs\Dược dân tộc 2

In [139]:
# Không chạy nữa
import os
import shutil
import pandas as pd

def copy_folders_from_dataframe(df, old_col, new_col):
    for index, row in df.iterrows():
        old_path = row[old_col]
        new_path = row[new_col]

        # Tạo thư mục mới nếu chưa tồn tại
        if not os.path.exists(new_path):
            os.makedirs(new_path)
        
        # Sao chép toàn bộ các file và thư mục con
        for item in os.listdir(old_path):
            src_path = os.path.join(old_path, item)
            dest_path = os.path.join(new_path, item)
            
            if os.path.isdir(src_path):
                shutil.copytree(src_path, dest_path, dirs_exist_ok=True)
            else:
                shutil.copy2(src_path, dest_path)
            
            print(f"Đã sao chép từ: {src_path} đến: {dest_path}")




In [ ]:
# Không chạy nữa
# Tên các cột trong DataFrame
old_col = 'Đường dẫn cũ'
new_col = 'Đường dẫn mới'

# Thực hiện sao chép từ DataFrame
copy_folders_from_dataframe(df_Entha_path_fix, old_col, new_col)

In [12]:
import os
import pandas as pd

def create_dataframe_from_directories(base_directory):
    data = []
    
    for family in os.listdir(base_directory):
        family_path = os.path.join(base_directory, family)
        if os.path.isdir(family_path):
            for genus in os.listdir(family_path):
                genus_path = os.path.join(family_path, genus)
                if os.path.isdir(genus_path):
                    for species in os.listdir(genus_path):
                        species_path = os.path.join(genus_path, species)
                        md_file = os.path.join(species_path, f'{species}.md')
                        if os.path.isfile(md_file):
                            data.append({'Họ': family, 'Chi': genus, 'Loài': species, 'Đường dẫn': md_file})
    
    # Tạo DataFrame từ danh sách dữ liệu
    df = pd.DataFrame(data, columns=['Họ', 'Chi', 'Loài', 'Đường dẫn'])
    
    return df

# Đường dẫn tới thư mục gốc chứa các thư mục
base_directory = r'path/to/Dược dân tộc'

# Tạo DataFrame
df = create_dataframe_from_directories(base_directory)

# Hiển thị DataFrame
print(df)


Index(['GENUS', 'SPECIES', 'FAMILY', 'Country_Enth'], dtype='object')

In [ ]:

# Đường dẫn tới thư mục gốc mới
root_directory_2 = r'..\docs\Dược dân tộc'

# Tạo cột 'Đường dẫn mới' bằng cách kết hợp đường dẫn gốc mới với cột 'FAMILY'
df_Entha_group['Path_Family'] = df_Entha_group.apply(lambda row: os.path.join(root_directory_2, row['FAMILY']), axis=1)
df_Entha_group['Path_GENUS'] = df_Entha_group.apply(lambda row: os.path.join(root_directory_2, row['FAMILY'], row['GENUS']), axis=1)
df_Entha_group['Path_GENUS'] = df_Entha_group.apply(lambda row: os.path.join(root_directory_2, row['FAMILY'], row['GENUS']), axis=1)